# Bird's Eye View (BEV) Visualization

This notebook walks through how modern **BEV perception** stacks work, with a focus on the **Lift-Splat-Shoot (LSS)** technique.

**What you will see:**
1. Load a pretrained BEV network (LSS) and run it on a sample surround-camera scene.
2. Visualize the *lift* step — the same feature map before and after being lifted into 3D.
3. Inspect the learned depth distribution, its expected value, and its uncertainty.
4. (Later parts) LiDAR-camera fusion, occupancy, and planning — all in the BEV grid.

> Reference: *Lift, Splat, Shoot: Encoding Images From Arbitrary Camera Rigs by Implicitly Unprojecting to 3D* — Philion & Fidler, ECCV 2020.

## Part 1 — Load a pretrained BEV Network (LSS)

We will:

1. Install dependencies and clone the official LSS repo.
2. Download the pretrained weights released by NVIDIA.
3. Build the `LiftSplatShoot` model and load the checkpoint.
4. Load a sample surround-camera input (6 cameras) and run the network.

> **Note:** the original LSS code assumes a specific directory layout. We mirror the inference setup from [`src/explore.py`](https://github.com/nv-tlabs/lift-splat-shoot/blob/master/src/explore.py) so that we can reuse NVIDIA's weights directly.

### 1.1 — Install dependencies

In [ ]:
!pip install pyquaternion
!pip install nuscenes-devkit tensorboardX efficientnet_pytorch==0.7.0

# Clone the official Lift-Splat-Shoot repo (NVIDIA Toronto AI Lab)
![ -d lift-splat-shoot ] || git clone --quiet https://github.com/nv-tlabs/lift-splat-shoot.git

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('lift-splat-shoot'))
print('LSS sources available at:', os.path.abspath('lift-splat-shoot'))

LSS sources available at: /content/lift-splat-shoot


### 1.2 — Download the pretrained LSS weights

NVIDIA hosts the checkpoint on Google Drive (see the LSS README). We fetch it with `gdown`. The file is ~200 MB and only needs to be downloaded once.

In [2]:
import os, gdown

WEIGHTS_PATH = 'model525000.pt'
LSS_WEIGHTS_URL = 'https://drive.google.com/uc?id=18fy-6beTFTZx5SrYLs9Xk7cY-fGSm7kw'

if not os.path.exists(WEIGHTS_PATH):
    try:
        gdown.download(LSS_WEIGHTS_URL, WEIGHTS_PATH, quiet=False)
    except Exception as e:
        print('Could not download pretrained weights:', e)
        print('Falling back to an ImageNet-initialised backbone later.')

print('Weights available:', os.path.exists(WEIGHTS_PATH))

Weights available: True


### 1.3 — Build the LSS model and load the checkpoint

The LSS model is built from two main pieces:

- `CamEncode` — an EfficientNet-B0 backbone that, for every pixel, predicts both a **feature vector** and a **categorical depth distribution** over a set of depth bins.
- `BevEncode` — a small ResNet that cleans up the rasterised BEV feature grid after the splat step.

In [3]:
import torch
from src.models import compile_model  # from the LSS repo

# These hyper-parameters match the settings NVIDIA used to train model525000.pt
grid_conf = {
    'xbound': [-50.0, 50.0, 0.5],   # BEV grid: x in [-50, 50] m, 0.5 m per cell
    'ybound': [-50.0, 50.0, 0.5],
    'zbound': [-10.0, 10.0, 20.0],
    'dbound': [4.0, 45.0, 1.0],     # 41 depth bins from 4 m to 45 m
}
data_aug_conf = {
    'resize_lim': (0.193, 0.225),
    'final_dim': (128, 352),        # network input size per camera
    'rot_lim': (-5.4, 5.4),
    'H': 900, 'W': 1600,
    'rand_flip': True,
    'bot_pct_lim': (0.0, 0.22),
    'cams': ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
             'CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT'],
    'Ncams': 6,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = compile_model(grid_conf, data_aug_conf, outC=1)  # outC=1 => drivable-area head

if os.path.exists(WEIGHTS_PATH):
    state = torch.load(WEIGHTS_PATH, map_location='cpu')
    model.load_state_dict(state)
    print('Loaded pretrained LSS weights.')
else:
    print('Using randomly initialised LSS (pretrained weights unavailable).')

model = model.to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model on {device} — {n_params/1e6:.1f} M parameters')

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 102MB/s] 
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded pretrained weights for efficientnet-b0
Loaded pretrained LSS weights.
Model on cuda — 14.3 M parameters


### 1.4 — Load a sample surround-camera input

LSS expects, for every one of the 6 cameras, a tensor of shape `(3, 128, 352)` together with the camera intrinsics and the camera → ego-vehicle extrinsics.

To keep the notebook self-contained we use the nuScenes **mini** split via `nuscenes-devkit` when it is available, and otherwise fall back to a synthetic scene with the canonical nuScenes camera rig. Either way the tensor shapes fed into the network are identical.

In [4]:
import numpy as np
from PIL import Image
import torchvision.transforms.functional as TF

CAMS = data_aug_conf['cams']
H, W = data_aug_conf['final_dim']  # 128 x 352

def synth_surround_scene(seed=0):
    """Generate a toy 6-camera scene so the notebook always runs.
    Each camera gets a sky/ground split with a few coloured 'vehicles'.
    Returns a list of PIL.Image of size (W, H)."""
    rng = np.random.default_rng(seed)
    imgs = []
    for i, cam in enumerate(CAMS):
        img = np.zeros((H, W, 3), dtype=np.uint8)
        # Sky gradient
        for y in range(H // 2):
            img[y] = (135 - y//2, 180 - y//3, 235)
        # Road
        img[H//2:] = (70, 70, 75)
        # Lane markings
        for lane_x in (W//3, 2*W//3):
            for y in range(H//2, H, 8):
                img[y:y+3, lane_x-1:lane_x+1] = 230
        # Random vehicles
        for _ in range(rng.integers(1, 4)):
            cx, cy = rng.integers(20, W-20), rng.integers(H//2+5, H-15)
            w, h = rng.integers(20, 50), rng.integers(10, 22)
            color = rng.integers(40, 220, size=3)
            img[max(0,cy-h):cy, max(0,cx-w//2):cx+w//2] = color
        imgs.append(Image.fromarray(img))
    return imgs

pil_images = synth_surround_scene(seed=7)

def pil_to_tensor(pil_img):
    # Same normalisation as the original LSS dataloader (src/data.py)
    t = TF.to_tensor(pil_img)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (t - mean) / std

imgs = torch.stack([pil_to_tensor(im) for im in pil_images])  # (6, 3, H, W)
print('imgs', tuple(imgs.shape))

imgs (6, 3, 128, 352)


In [5]:
# Canonical nuScenes camera rig (rough values — good enough for visualisation).
# Yaw angles (deg) of each camera around the ego-Z axis, and their (x, y, z) offsets (m) in ego frame.
RIG = {
    'CAM_FRONT_LEFT':  (dict(yaw= 55.0, xyz=( 1.52,  0.50, 1.50))),
    'CAM_FRONT':       (dict(yaw=  0.0, xyz=( 1.72,  0.00, 1.50))),
    'CAM_FRONT_RIGHT': (dict(yaw=-55.0, xyz=( 1.52, -0.50, 1.50))),
    'CAM_BACK_LEFT':   (dict(yaw=110.0, xyz=( 1.04,  0.48, 1.50))),
    'CAM_BACK':        (dict(yaw=180.0, xyz=( 0.05,  0.00, 1.50))),
    'CAM_BACK_RIGHT':  (dict(yaw=-110.0,xyz=( 1.04, -0.48, 1.50))),
}

def intrinsics(fx=1266.0, fy=1266.0, cx=816.0, cy=491.0):
    K = torch.tensor([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=torch.float32)
    return K

def yaw_matrix(deg):
    a = np.deg2rad(deg)
    c, s = np.cos(a), np.sin(a)
    return torch.tensor([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=torch.float32)

# Build the tensors LSS expects: intrins, rots, trans, post_rots, post_trans
intrins, rots, trans = [], [], []
for cam in CAMS:
    intrins.append(intrinsics())
    rots.append(yaw_matrix(RIG[cam]['yaw']))
    trans.append(torch.tensor(RIG[cam]['xyz'], dtype=torch.float32))
intrins = torch.stack(intrins)
rots    = torch.stack(rots)
trans   = torch.stack(trans)

# Post-augmentation transforms: identity (we already resized/normalised the images).
post_rots  = torch.eye(3).unsqueeze(0).repeat(6, 1, 1)
post_trans = torch.zeros(6, 3)

# Add the batch dimension expected by the model
batch = [x.unsqueeze(0).to(device) for x in (imgs, rots, trans, intrins, post_rots, post_trans)]
for name, t in zip(['imgs','rots','trans','intrins','post_rots','post_trans'], batch):
    print(f'{name:11s} {tuple(t.shape)}')

imgs        (1, 6, 3, 128, 352)
rots        (1, 6, 3, 3)
trans       (1, 6, 3)
intrins     (1, 6, 3, 3)
post_rots   (1, 6, 3, 3)
post_trans  (1, 6, 3)


### 1.4b — Swap in a **real** nuScenes surround sample

We now replace the toy images with one real keyframe (6 cameras) from the `v1.0-mini` split of nuScenes. The mini split is ~4 GB — it's the smallest public nuScenes bundle that ships with complete metadata/calibration. Download is a one-off; the tarball is cached and extraction is skipped on re-runs.

**What happens here:**
1. Download & extract `v1.0-mini.tgz` (skipped if already present — or if you manually upload the tarball to Colab).
2. Instantiate the official LSS `SegmentationData` loader — this guarantees the calibration (`rots`, `trans`, `intrins`, `post_rots`, `post_trans`) is **exactly** what the pretrained weights expect.
3. Overwrite the synthetic `pil_images` + `batch` tensors with the real sample. All downstream cells (visualize / Part 2) keep working as-is.

In [ ]:
import os, subprocess, sys

NUSCENES_ROOT = './nuscenes-mini'
TARBALL = '/tmp/v1.0-mini.tgz'

META_READY = os.path.exists(os.path.join(NUSCENES_ROOT, 'v1.0-mini', 'scene.json'))

if not META_READY:
    if not (os.path.exists(TARBALL) and os.path.getsize(TARBALL) > 10_000_000):
        # Try the official download URL. The nuScenes terms page has a direct
        # tarball link — if the URL ever changes, you can also upload the
        # tarball manually to /tmp/v1.0-mini.tgz and re-run this cell.
        CANDIDATE_URLS = [
            'https://www.nuscenes.org/data/v1.0-mini.tgz',
            'https://d36yt3mvayqw5m.cloudfront.net/public/v1.0/v1.0-mini.tgz',
            'https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-mini.tgz',
        ]
        for url in CANDIDATE_URLS:
            print(f'Trying {url} …')
            rc = subprocess.call(['wget', '-q', '--show-progress', '-c', url, '-O', TARBALL])
            if rc == 0 and os.path.getsize(TARBALL) > 10_000_000:
                print('Downloaded from', url)
                break
        else:
            print('\nAuto-download failed. Please manually download v1.0-mini.tgz from')
            print('    https://www.nuscenes.org/download')
            print(f'and upload it to {TARBALL} in the Colab file tree, then re-run this cell.')

    if os.path.exists(TARBALL) and os.path.getsize(TARBALL) > 10_000_000:
        os.makedirs(NUSCENES_ROOT, exist_ok=True)
        print('Extracting …')
        subprocess.check_call(['tar', '-xzf', TARBALL, '-C', NUSCENES_ROOT])

META_READY = os.path.exists(os.path.join(NUSCENES_ROOT, 'v1.0-mini', 'scene.json'))
print('nuScenes metadata ready:', META_READY)


In [ ]:
# Use the official LSS dataloader to build one batch from nuScenes mini.
# This matches the exact preprocessing NVIDIA used when training model525000.pt.
from PIL import Image

if META_READY:
    from nuscenes.nuscenes import NuScenes
    from src.data import SegmentationData

    nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)
    # is_train=False => deterministic augmentation (center crop, no flip/rotate)
    ds = SegmentationData(nusc, is_train=False,
                          data_aug_conf=data_aug_conf, grid_conf=grid_conf)

    SAMPLE_IDX = 0   # try 1, 2, … to jump to other keyframes in the mini split
    imgs_r, rots_r, trans_r, intrins_r, post_rots_r, post_trans_r, binimg_r = ds[SAMPLE_IDX]

    # Convert the already-augmented model-input tensor back to a PIL image so
    # that overlays / scatter plots in later cells line up with the feature grid.
    _MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    _STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    def tensor_to_pil(t):
        x = (t.cpu() * _STD + _MEAN).clamp(0, 1)
        return Image.fromarray((x.permute(1, 2, 0).numpy() * 255).astype('uint8'))

    pil_images = [tensor_to_pil(imgs_r[i]) for i in range(6)]
    imgs  = imgs_r
    batch = [x.unsqueeze(0).to(device) for x in
             (imgs_r, rots_r, trans_r, intrins_r, post_rots_r, post_trans_r)]

    # Keep the ground-truth BEV mask around for later comparison.
    binimg_gt = binimg_r.cpu().numpy()

    rec = ds.ixes[SAMPLE_IDX]
    scene_name = nusc.get('scene', rec['scene_token'])['name']
    print(f'Loaded real nuScenes sample #{SAMPLE_IDX} from scene {scene_name!r}')
    for name, t in zip(['imgs','rots','trans','intrins','post_rots','post_trans'], batch):
        print(f'  {name:11s} {tuple(t.shape)}')
else:
    binimg_gt = None
    print('nuScenes mini not available — keeping the synthetic scene.')


### 1.5 — Run the network and visualize the inputs + BEV output

In [6]:
import matplotlib.pyplot as plt

with torch.no_grad():
    bev_logits = model(*batch)          # (1, outC, 200, 200)
    bev = torch.sigmoid(bev_logits)[0, 0].cpu().numpy()

# Show the 6 input cameras + the BEV prediction
order = ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
         'CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT']
fig, axes = plt.subplots(3, 3, figsize=(13, 7))
for ax, cam in zip(axes[:2].flatten(), order):
    ax.imshow(pil_images[CAMS.index(cam)])
    ax.set_title(cam, fontsize=9); ax.axis('off')

for ax in axes[2]:
    ax.axis('off')
bev_ax = fig.add_subplot(3, 1, 3)
bev_ax.imshow(bev, origin='lower', cmap='magma', extent=[-50, 50, -50, 50])
bev_ax.scatter([0], [0], c='cyan', marker='^', s=80, label='ego')
bev_ax.set_xlabel('x [m]'); bev_ax.set_ylabel('y [m]')
bev_ax.set_title('LSS BEV output (drivable-area score)')
bev_ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

print('BEV tensor:', bev.shape, 'range:', float(bev.min()), '→', float(bev.max()))

BEV tensor: (200, 200) range: 0.0019988843705505133 → 0.960487961769104


## Part 2 — Visualizing the **Lift** step

> **Heads up on the input data.** The previous cells use a synthetic surround-camera scene so the notebook can run anywhere. NVIDIA's pretrained LSS weights *are* loaded, but the depth-net has never seen toy images like this, so the predicted depths/BEV mask are not meaningful. The *mechanics* of the lift, however, are identical — which is exactly what we want to visualize here. To swap in a real nuScenes sample, install `nuscenes-devkit`, download the `v1.0-mini` split, and replace the `pil_images` / `rots` / `trans` / `intrins` tensors with the ones returned by `NuscData.__getitem__` (see `lift-splat-shoot/src/data.py`).

The lift turns every pixel feature into a *ray of features* — one per depth bin — weighted by a learned categorical distribution over depth. Concretely:

```
for each pixel (u, v):
    f(u, v)      ∈ R^C            # context feature vector
    α(u, v, d)   ∈ Δ^D            # softmax over D depth bins
    lifted(u, v, d) = α(u, v, d) · f(u, v)    ∈ R^C
```

We now extract these intermediate tensors from the pretrained `CamEncode` module.

In [ ]:
# Extract intermediate tensors from the CamEncode module for all 6 cameras
imgs_b, rots_b, trans_b, intrins_b, post_rots_b, post_trans_b = batch
B, N, _, imH, imW = imgs_b.shape
imgs_flat = imgs_b.view(B * N, 3, imH, imW)

with torch.no_grad():
    eff_feat     = model.camencode.get_eff_depth(imgs_flat)   # (N, 512, fH, fW)
    depth_logits = model.camencode.depthnet(eff_feat)         # (N, D+C, fH, fW)
    D_ = model.camencode.D
    C_ = model.camencode.C
    depth_dist = depth_logits[:, :D_].softmax(dim=1)          # (N, D, fH, fW)
    ctx_feat   = depth_logits[:, D_:]                         # (N, C, fH, fW)
    lifted     = depth_dist.unsqueeze(1) * ctx_feat.unsqueeze(2)  # (N, C, D, fH, fW)

fH, fW = eff_feat.shape[-2:]
print(f'Feature grid per camera : {fH} x {fW}   (input {imH}x{imW}, stride {imH//fH})')
print(f'Depth bins              : D = {D_}  from {grid_conf["dbound"][0]} m to {grid_conf["dbound"][1]} m')
print(f'Context feature channels: C = {C_}')
print('')
print(f'2D features before lift : ctx_feat    {tuple(ctx_feat.shape)}')
print(f'Depth distribution      : depth_dist  {tuple(depth_dist.shape)}')
print(f'Lifted 3D frustum feats : lifted      {tuple(lifted.shape)}   # = depth_dist ⊗ ctx_feat')


### 2.1 — Features **before** the lift (2D feature maps)

The `CamEncode` module first runs an EfficientNet-B0 trunk and produces a dense feature map of shape `(C=64, fH=8, fW=22)` for each camera image. We project those 64-channel features down to 3 channels with PCA so we can visualize them as RGB.

Brighter / more saturated regions correspond to parts of the image the network finds informative.

In [ ]:
from sklearn.decomposition import PCA

def feat_to_rgb(feat_2d):
    """(C, H, W) tensor -> (H, W, 3) uint8 image via per-pixel PCA."""
    C_, H_, W_ = feat_2d.shape
    X = feat_2d.reshape(C_, -1).T.cpu().numpy()      # (H*W, C)
    Y = PCA(n_components=3).fit_transform(X)         # (H*W, 3)
    Y = (Y - Y.min(0)) / (Y.max(0) - Y.min(0) + 1e-6)
    return Y.reshape(H_, W_, 3)

fig, axes = plt.subplots(2, 6, figsize=(16, 4.2))
for i, cam in enumerate(CAMS):
    axes[0, i].imshow(pil_images[i]);                   axes[0, i].axis('off')
    axes[0, i].set_title(cam, fontsize=8)
    axes[1, i].imshow(feat_to_rgb(ctx_feat[i]));         axes[1, i].axis('off')
axes[0, 0].set_title('input\n' + CAMS[0], fontsize=8)
axes[1, 0].set_title('PCA features', fontsize=8, loc='left')
plt.suptitle('2D context features BEFORE the lift (PCA → RGB)', y=1.02)
plt.tight_layout(); plt.show()


### 2.2 — Depth distribution per pixel

For every pixel in the downsampled grid, LSS predicts a categorical distribution over `D = 41` depth bins (from 4 m to 45 m, every 1 m). Three useful summaries:

- **argmax depth** — the most-likely depth bin per pixel.
- **expected depth** — `E[d] = Σ_d d · p(d)` — softer estimate.
- **entropy** — `H = -Σ_d p(d) log p(d)` — *uncertainty*. High entropy ≈ the network is unsure where this pixel lives in 3D.

In [ ]:
cam_idx = CAMS.index('CAM_FRONT')
depth_bins = torch.arange(*grid_conf['dbound'], dtype=torch.float32, device=device)  # (D,)
dd = depth_dist[cam_idx]                                                     # (D, fH, fW)
argmax_depth   = depth_bins[dd.argmax(0)]
expected_depth = (dd * depth_bins.view(-1, 1, 1)).sum(0)
entropy        = -(dd * (dd + 1e-12).log()).sum(0)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
axes[0].imshow(pil_images[cam_idx]); axes[0].set_title(f'{CAMS[cam_idx]} input'); axes[0].axis('off')
for ax, data, title, cmap in [
    (axes[1], argmax_depth,   'argmax depth [m]',        'turbo'),
    (axes[2], expected_depth, 'expected depth E[d] [m]', 'turbo'),
    (axes[3], entropy,        'entropy (uncertainty)',   'magma'),
]:
    im = ax.imshow(data.cpu(), cmap=cmap); ax.set_title(title); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout(); plt.show()


### 2.3 — Depth PDFs at individual pixels

Rather than collapsing the distribution to a single number, let's plot the full PDF at a few sampled pixels. Far-away pixels (upper part of the feature map, near the horizon) typically have broader distributions, whereas pixels on the road surface usually spike at a specific depth.

In [ ]:
sample_pixels = [
    (fH // 4,     fW // 2),   # upper-middle (far)
    (fH // 2,     fW // 3),   # mid-left
    (3 * fH // 4, 2 * fW // 3),  # lower-right (close)
]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4))
ax0.imshow(pil_images[cam_idx]); ax0.set_title(f'{CAMS[cam_idx]} — sampled pixels'); ax0.axis('off')
stride_y, stride_x = imH // fH, imW // fW
for (yy, xx) in sample_pixels:
    ax0.scatter([xx * stride_x + stride_x/2], [yy * stride_y + stride_y/2], s=120, edgecolor='white', linewidth=2)
    ax1.plot(depth_bins.cpu(), dd[:, yy, xx].cpu(), marker='o', label=f'pixel ({yy}, {xx})')
ax1.set_xlabel('depth [m]'); ax1.set_ylabel('P(depth)')
ax1.set_title('Categorical depth distribution at each marked pixel')
ax1.legend(); ax1.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 2.4 — The lifted 3D frustum

Each camera's feature map is *lifted* into a 3D frustum of shape `(D, fH, fW)`, one voxel per (depth, pixel) combination. Using the camera intrinsics and ego-frame extrinsics, `model.get_geometry(...)` returns the `(x, y, z)` location of every voxel in the ego frame.

We plot the top-magnitude voxels in 3D, colored by the weight of the lifted feature (higher = more confident the network placed features here).

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3d projection)

with torch.no_grad():
    geom = model.get_geometry(rots_b, trans_b, intrins_b, post_rots_b, post_trans_b)
# geom: (B, N, D, fH, fW, 3) in the ego frame
geom_cam = geom[0, cam_idx].cpu().numpy()                        # (D, fH, fW, 3)
weights  = lifted[cam_idx].norm(dim=0).cpu().numpy()             # (D, fH, fW)

# Keep only the top-magnitude voxels so the scatter is legible
thresh = np.quantile(weights, 0.90)
mask = weights > thresh
pts   = geom_cam[mask]
vals  = weights[mask]

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=vals, cmap='plasma', s=6, alpha=0.65)
ax.scatter([0], [0], [0], c='cyan', marker='^', s=120, edgecolor='black', label='ego')
ax.set_xlabel('x (forward) [m]'); ax.set_ylabel('y (left) [m]'); ax.set_zlabel('z [m]')
ax.set_title(f'{CAMS[cam_idx]} frustum AFTER the lift (top 10% voxels)')
ax.view_init(elev=22, azim=-70)
ax.legend(); plt.colorbar(sc, ax=ax, fraction=0.03, label='||lifted feature||')
plt.tight_layout(); plt.show()


### 2.5 — The **splat** — pooling all 6 cameras into one BEV grid

The final step is *splat*: every lifted voxel is scattered into its corresponding cell of a 2D BEV grid. Voxels that fall into the same cell are summed (the "cumulative-sum trick" in the LSS paper). This gives a single feature tensor `(C, 200, 200)` covering a 100 m × 100 m region around the ego vehicle.

We show two views of that tensor:
1. the L2 magnitude of the per-cell feature vector (where does LSS place *any* feature),
2. a PCA → RGB projection (what kind of features were placed where).

In [ ]:
with torch.no_grad():
    x_cam_feats = model.get_cam_feats(imgs_b)                    # (B, N, D, fH, fW, C)
    bev_feat    = model.voxel_pooling(geom, x_cam_feats)          # (B, C, X, Y)

bev_mag = bev_feat[0].norm(dim=0).cpu().numpy()                  # (X, Y)
bev_rgb = feat_to_rgb(bev_feat[0].cpu())                          # (X, Y, 3)

# LSS convention: axis 0 = x (forward), axis 1 = y (left). Transpose so y is horizontal, x is vertical-up.
def to_topdown(arr):
    return np.flipud(arr.transpose(1, 0) if arr.ndim == 2 else arr.transpose(1, 0, 2))

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
extent = [-50, 50, -50, 50]
axes[0].imshow(to_topdown(bev_mag), cmap='magma', extent=extent)
axes[0].set_title('BEV feature magnitude ||f||₂')
axes[1].imshow(to_topdown(bev_rgb),                extent=extent)
axes[1].set_title('BEV features (PCA → RGB)')
for ax in axes:
    ax.scatter([0], [0], c='cyan', marker='^', s=120, edgecolor='black', label='ego')
    ax.set_xlabel('y [m]  (left +)'); ax.set_ylabel('x [m]  (forward +)')
    ax.legend(loc='upper right')
plt.suptitle('Splatted BEV features from all 6 cameras', y=1.02)
plt.tight_layout(); plt.show()


**Recap of Part 2.** We walked through every stage of a Lift-Splat-Shoot forward pass:

| Stage | Tensor shape | What it means |
|------|--------------|---------------|
| 2D image features | `(N, C, fH, fW)` | per-pixel context descriptor |
| Depth distribution | `(N, D, fH, fW)` | `p(depth)` per pixel |
| Lifted 3D frustum | `(N, C, D, fH, fW)` | outer product `α ⊗ f` |
| Voxel-pooled BEV | `(1, C, 200, 200)` | everything summed into one BEV grid |

Up next — **Part 3**: using these same building blocks to look at depth *uncertainty* across the scene and compare it against ground truth when available. Then Parts 4–6 cover LiDAR-camera fusion, occupancy, and planning, all in the same BEV grid.

---

**Next parts (to be added):**
- Part 2 — Visualize the *lift*: features before and after being raised to 3D.
- Part 3 — Depth distribution: expected depth, entropy, per-pixel PDFs.
- Part 4 — LiDAR-camera fusion in the BEV grid.
- Part 5 — Occupancy prediction (multi-height BEV slices).
- Part 6 — Planning on a BEV cost map.

## Part 3 — LiDAR-camera Fusion in the BEV Grid

> **Quick note on your question.** You're right to check — **LSS never lifts pixels, only feature vectors.** The `ctx_feat` tensor we PCA'd in section 2.1 (C=64 channels at 8×22 resolution) is exactly what gets outer-producted with the depth distribution. The 3D frustum in 2.4 contains `D · fH · fW = 41 · 8 · 22 ≈ 7,200` voxels per camera — one per *feature-map* location, not per pixel of the 128×352 input. The raw image is only shown for our eyes.

Cameras are great at semantics but bad at metric depth. LiDAR is the opposite — exact geometry, zero appearance. **BEV is the natural meeting point**: each modality ends up as a tensor of shape `(C, 200, 200)` indexed by the same ego-frame cells, so fusion is a matter of stacking channels.

In this part we:
1. Pull the `LIDAR_TOP` point cloud from the same nuScenes keyframe, transform it into the ego frame, and rasterize it to BEV "pillars" (density, max-height, mean-reflectance).
2. Look at camera BEV (from LSS) and LiDAR BEV side-by-side, and overlay the raw LiDAR points on top of the LSS feature magnitudes.
3. Do the simplest possible fusion — channel-wise concatenation, à la BEVFusion — and visualize the combined feature grid.

In [ ]:
# --- BEV display helper -------------------------------------------------
# Convention: x forward (vertical, + up), y left (horizontal, + left).
BEV_EXTENT = [-50, 50, -50, 50]

def bev_show(ax, arr, **kw):
    kw.setdefault('extent', BEV_EXTENT)
    im = ax.imshow(arr, origin='lower', **kw)
    ax.invert_xaxis()                # make +y (ego left) appear on the LEFT
    ax.set_xlabel('y [m]   (+ = left of ego)')
    ax.set_ylabel('x [m]   (+ = forward)')
    ax.set_aspect('equal')
    return im


### 3.1 — Pull the LiDAR point cloud for the same keyframe

We load `LIDAR_TOP` (Velodyne HDL-32), rotate + translate it into the ego frame using the calibrated-sensor record, and inspect its extent.

In [ ]:
if META_READY:
    from nuscenes.utils.data_classes import LidarPointCloud
    from pyquaternion import Quaternion

    rec       = ds.ixes[SAMPLE_IDX]
    lidar_sd  = nusc.get('sample_data', rec['data']['LIDAR_TOP'])
    calib     = nusc.get('calibrated_sensor', lidar_sd['calibrated_sensor_token'])

    pc = LidarPointCloud.from_file(os.path.join(nusc.dataroot, lidar_sd['filename']))
    pc.rotate(Quaternion(calib['rotation']).rotation_matrix)
    pc.translate(np.array(calib['translation']))

    lidar_xyz       = pc.points[:3].T.astype(np.float32)   # (N, 3) in EGO frame
    lidar_intensity = pc.points[3].astype(np.float32)
else:
    # Synthetic cloud so this part still runs when nuScenes isn't available
    rng = np.random.default_rng(0); N = 30_000
    lidar_xyz = np.column_stack([rng.uniform(-50, 50, N),
                                 rng.uniform(-50, 50, N),
                                 rng.uniform(-2.2, 2.5, N)]).astype(np.float32)
    lidar_intensity = rng.uniform(0, 1, N).astype(np.float32)

print(f'{lidar_xyz.shape[0]:,} LiDAR points in ego frame')
print(f'  x range: {lidar_xyz[:,0].min():6.1f} .. {lidar_xyz[:,0].max():6.1f} m (forward)')
print(f'  y range: {lidar_xyz[:,1].min():6.1f} .. {lidar_xyz[:,1].max():6.1f} m (left)')
print(f'  z range: {lidar_xyz[:,2].min():6.1f} .. {lidar_xyz[:,2].max():6.1f} m (up)')


### 3.2 — Rasterize LiDAR to a BEV "pillar" grid

For each BEV cell we collapse the vertical dimension into three fast, hand-crafted summary channels:

- **point density** (how many returns hit the cell — indicates surfaces),
- **max-z** (height of the tallest point — reveals poles, trees, buildings),
- **mean reflectance** (proxy for material — lane markings are bright).

This is essentially the non-learned version of what PointPillars / VoxelNet do before their PFN.

In [ ]:
def lidar_to_bev(xyz, intensity, grid_conf):
    xmin, xmax, xres = grid_conf['xbound']
    ymin, ymax, yres = grid_conf['ybound']
    zmin, zmax, _    = grid_conf['zbound']
    X = int(round((xmax - xmin) / xres))
    Y = int(round((ymax - ymin) / yres))

    m = ((xyz[:, 0] >= xmin) & (xyz[:, 0] < xmax) &
         (xyz[:, 1] >= ymin) & (xyz[:, 1] < ymax) &
         (xyz[:, 2] >= zmin) & (xyz[:, 2] < zmax))
    xyz, intensity = xyz[m], intensity[m]

    xi = ((xyz[:, 0] - xmin) / xres).astype(np.int32)
    yi = ((xyz[:, 1] - ymin) / yres).astype(np.int32)

    density = np.zeros((X, Y), np.float32); np.add.at(density,  (xi, yi), 1)
    mean_i  = np.zeros((X, Y), np.float32); np.add.at(mean_i,   (xi, yi), intensity)
    max_z   = np.full ((X, Y), zmin, np.float32); np.maximum.at(max_z, (xi, yi), xyz[:, 2])
    mean_i  = np.where(density > 0, mean_i / np.maximum(density, 1.0), 0.0)
    return density, max_z, mean_i

density, max_z, mean_i = lidar_to_bev(lidar_xyz, lidar_intensity, grid_conf)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
im = bev_show(axes[0], np.log1p(density), cmap='magma'); axes[0].set_title('point density  (log)')
plt.colorbar(im, ax=axes[0], fraction=0.04)
im = bev_show(axes[1], max_z, cmap='viridis'); axes[1].set_title('max point height [m]')
plt.colorbar(im, ax=axes[1], fraction=0.04)
im = bev_show(axes[2], mean_i, cmap='plasma'); axes[2].set_title('mean reflectance')
plt.colorbar(im, ax=axes[2], fraction=0.04)
for ax in axes:
    ax.scatter([0], [0], c='cyan', marker='^', s=100, edgecolor='black', label='ego')
plt.suptitle('LiDAR rasterized into the same BEV grid as LSS', y=1.02)
plt.tight_layout(); plt.show()


### 3.3 — Camera BEV  ↔  LiDAR BEV  (same grid)

Both modalities live on the same 200×200 BEV grid at 0.5 m/cell. Side-by-side you can see how complementary they are: camera features light up where things *look* different (road markings, parked cars, facades); LiDAR lights up where things *are* (every physical return).

In [ ]:
camera_bev_mag = bev_feat[0].norm(dim=0).cpu().numpy()    # (X, Y)

fig, axes = plt.subplots(1, 3, figsize=(17, 5.8))
im = bev_show(axes[0], camera_bev_mag,      cmap='magma'); plt.colorbar(im, ax=axes[0], fraction=0.04)
axes[0].set_title('Camera BEV (LSS)  —  ||features||₂')

im = bev_show(axes[1], np.log1p(density),   cmap='magma'); plt.colorbar(im, ax=axes[1], fraction=0.04)
axes[1].set_title('LiDAR BEV  —  log point density')

# Overlay: camera BEV in grayscale + raw LiDAR points, colored by height.
bev_show(axes[2], camera_bev_mag, cmap='gray', alpha=0.85)
step = max(1, lidar_xyz.shape[0] // 20000)
sub = lidar_xyz[::step]
sc = axes[2].scatter(sub[:, 1], sub[:, 0], c=sub[:, 2], s=0.6,
                     cmap='turbo', vmin=-2, vmax=3, alpha=0.7)
axes[2].set_title('Camera BEV  +  raw LiDAR points (z-colored)')
plt.colorbar(sc, ax=axes[2], fraction=0.04, label='point z [m]')
for ax in axes:
    ax.scatter([0], [0], c='cyan', marker='^', s=100, edgecolor='black')
plt.tight_layout(); plt.show()


### 3.4 — BEVFusion-style fusion: just **concat the channels**

This is the core insight of [BEVFusion (Liu et al., 2022)](https://arxiv.org/abs/2205.13542): once every modality produces features on the same BEV grid, fusion is literally `torch.cat(..., dim=1)` followed by a small conv head. Here we do the cat and visualize the result with a shared PCA basis so camera / LiDAR / fused live in the same color space.

In [ ]:
# Make a 3-channel LiDAR "feature" tensor (density / height / reflectance),
# scaled to roughly the same dynamic range as the camera features.
lidar_bev = torch.from_numpy(np.stack([
    np.log1p(density) / 6.0,
    (max_z - max_z.min()) / (max_z.max() - max_z.min() + 1e-6),
    mean_i / (mean_i.max() + 1e-6),
])).float()

camera_bev = bev_feat[0].cpu()                              # (64, X, Y)
fused_bev  = torch.cat([camera_bev, lidar_bev], dim=0)      # (67, X, Y)
print(f'camera {tuple(camera_bev.shape)} ⊕ lidar {tuple(lidar_bev.shape)}  →  fused {tuple(fused_bev.shape)}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, feats, title in zip(
        axes,
        [camera_bev, lidar_bev, fused_bev],
        ['Camera-only  (PCA)', 'LiDAR-only  (PCA)', 'Fused  (PCA)']):
    bev_show(ax, feat_to_rgb(feats))
    ax.scatter([0], [0], c='cyan', marker='^', s=100, edgecolor='black')
    ax.set_title(title)
plt.suptitle('Channel-wise concatenation of camera and LiDAR BEV features (BEVFusion-style)', y=1.02)
plt.tight_layout(); plt.show()


**Take-aways for Part 3.**

- BEV is a *shared address space*: every modality, once lifted, is a `(C, 200, 200)` tensor indexed the same way — so fusion is trivially `torch.cat` on the channel dim.
- Camera features activate strongly on appearance cues (lane markings, parked vehicles, facades). LiDAR lights up on every physical return and is metrically exact.
- Real BEVFusion puts a learned PointPillars / VoxelNet backbone on the LiDAR side and a shared conv head after the concat — the concat *is* the fusion; the backbone is what extracts useful channels from each modality first.

Next up:
- **Part 4 — Occupancy** on a voxelized BEV grid, with multi-height slice views.
- **Part 5 — Planning**: turning a BEV cost map into an ego trajectory.